# **Introduction (Assignment 1, Deadline 23/10/2026 at 08:00 AM)**

Students write text here, explaining:
 - What it means to preprocess data and what is a preprocessing pipeline
   Lior's answer: In TMS-EEG, preprocessing is the act of separating the true neural signal generated by the cortex in response to a TMS pulse from noise, which is electrical activity originating from non-brain sources such as eye movements.
Since there is no single step for artifact removal, we use a preprocessing pipeline—a structured sequence of systematic data transformations. These pipelines are generally divided into two main approaches:
1 Conservationist approaches: Preserve the raw signal as much as possible to retain genuine cortical activity.
2 Interventionist approaches: Apply aggressive transformations to eliminate all non-brain artifacts, assuming the true neural signal remains uninterpretable otherwise.
 - What are the main artefacts that can be found in the TMS-EEG signal and their causes
   lior's answer:
1 TMS Pulse Artifact: The magnetic field emitted by the TMS produces a massive electrical signal in the recording system. The entire brain signal appears completely flat, with only the pulse remaining prominent.
2 Ocular / Eye Blink Artifacts: The EEG device picks up eye movements / blinks, and additionally, TMS stimulation in frontal areas triggers this. Sharp jumps in amplitude in frontal electrodes.
3 Muscle Artifacts: Such as facial and jaw muscles. Sometimes with a TMS pulse we can activate them if we get close to speech areas. High amplitudes with very high frequency that interfere with seeing brain activity.
 - **Optional**: a personal comment or opinion on the problems posed by preprocessing, based on what you heard in class
   lior's answer: In my opinion, the main challenge posed by preprocessing is that the two primary approaches represent are extreme philosophies. On the one hand, a strictly conservationist approach protects brain markers at all costs, leaving the data contaminated with severe artifacts that obscure the neural signal. On the other hand, a purely interventionist approach is overly aggressive, risking the deletion of genuine brain activity along with the noise. In TMS-EEG, this is particularly problematic because removing a high-amplitude artifact might inadvertently eliminate the very physiological response we are investigating. Ultimately, preprocessing requires a balanced, flexible approach tailored to the specific experimental design rather than adhering rigidly to one extreme.

In [1]:
from pathlib import Path                            # to build a bridge between Python and the filesystem 

import numpy as np                                  # to perform array operations
import matplotlib                                   # this and the following to draw plots
import matplotlib.pyplot as plt
matplotlib.use('Qt5Agg')                            # to make plots interactive

import mne                                          # to read and manipulate EEG data

from scripts import utils                           # custom functions to shorten the code in this notebook

Waiting for Loading-Thread to finish... (max. 10 sec)
Channels marked as bad:
none
Attempting to create new mne-python configuration file:
C:\Users\user\.mne\mne-python.json
Could not read the C:\Users\user\.mne\mne-python.json json file during the writing. Assuming it is empty. Got: Expecting value: line 1 column 1 (char 0)
Channels marked as bad:
none


# **1. Basic Preprocessing (Assignment 1, Deadline 23/10/2026 at 08:00 AM)**

Students write text here, listing the operations that we categorise as "basic preprocessing".
If you cannot remember what the steps are, look at the code in the cells below and you will recall.
Lior's answer:
1.Loading files: signal file (.eeg), metadata file (.vhdr), and marker file (.vmrk).
2.Checking data and recording characteristics: sampling rate, duration, channels, and initial identification of noise level.
3.Adjusting and cleaning channels: removing non-brain channels like EOG that are not required for brain analysis.
4.Setting electrode location: linking channel names onto a head template.


## **1.1. Loading the Data**

**Optional but welcome:** students write here any considerations about this step.
Lior's answer: In this step, we load the raw EEG recording files into memory so that the software can process them. All this code does is search for the file in the folder and load it into memory using the command.

In [2]:
data_dir = Path("data")
subject_and_condition = "S02C1_M1"

file_of_interest = list(data_dir.glob(pattern=f"{subject_and_condition}.vhdr"))
print(file_of_interest, "|", type(file_of_interest), "|", file_of_interest[0])

[WindowsPath('data/S02C1_M1.vhdr')] | <class 'list'> | data\S02C1_M1.vhdr


In [3]:
eeg_data = mne.io.read_raw_brainvision(vhdr_fname=file_of_interest[0])

Extracting parameters from data\S02C1_M1.vhdr...
Setting channel info structure...


## **1.2. Dropping Non-EEG Channels**

**Compulsory:** students write here the functional significance of this step (that is, why we do it). Lior's answer: We identify and drop the non-brain channels (such as the EOG eye channel) because they do not reflect cortical activity, ensuring that our analysis focuses exclusively on EEG data.

In [4]:
channels_to_drop = []

for channel_name in eeg_data.ch_names:
    try:
        int(channel_name[-1])
    except ValueError:
        if channel_name[-1] != "z": 
            channels_to_drop.append(channel_name)
print(f"Channels to drop: {channels_to_drop}")

eeg_data = eeg_data.drop_channels(ch_names=channels_to_drop)

Channels to drop: ['EOG', 'FDI']


## **1.3. Setting the Channel Montage**

**Compulsory:** students write here the functional significance of this step (that is, why we do it). Lior's answer: In this step, we assign 3D coordinates to each EEG channel using a standard head template (EasyCAP M1). We do this to know where each electrode is located on the scalp, which is necessary for making topographic maps (topoplots).

In [27]:
easycap_m1_montage = mne.channels.make_standard_montage(kind="easycap-M1")
easycap_m1_montage.plot()
eeg_data.set_montage(montage=easycap_m1_montage,
                     on_missing="ignore")

<RawBrainVision | S02C1_M1.eeg, 70 x 3307600 (661.5 s), ~86 KiB, data not loaded>


**Compulsory:** here, students describe and comment the plot generated by the code below. In particular, I would like to read:

- What each row represents
- What are the $x$ and $y$ of each row, and their units of measurement
- A description of the artefacts you see, and their likely causes

Lior's answer:
In this graph: each row stands for the specific electrode that was placed on the patient's head. For example, electrodes that was placed on the frontal part of our brain start with the letter F. The x-axis is Time and the unit of measure in seconds. The Y-axis is Electric voltage measured in microvolts.
In this graph we can observe:
1 Poor electrode contact with the brain (for example, F7)
2 Eye Blink Artifact (mostly in the frontal areas) 
3 Muscle Artifacts (high amplitude and high frequency that affect many channels).
4 TMS Pulse Artifact (after stimulation).
 


In [26]:
eeg_data.plot()

**Compulsory:** here, students define event markers and explain why they are important for EEG data analysis
Lior's asnwer: Event markers are timestamps that mark exactly when a specific event occurred during the experiment, and without them EEG is just a long, continuous sequence of signals. They allow us to know exactly when stimuli were delivered and to cut the signal into segments around them, thus focusing on analyzing brain responses to the specific stimulus or removing the TMS artifact.

**Optional:** any comments that go beyond a mere definition are welcome. For example, you could make comments about the structure of event arrays and what it tells us about the nature of events
Lior's answer: The MNE event array is a matrix with 3 columns that organizes the event markers for analysis: 
1.	Time of occurrence: The exact position of the event measured in sample index/array units. 
2.	Duration: The length of the event in sample points (usually 1 for an instantaneous TMS pulse). 
3.	Event Code: The numeric ID representing the type of event (e.g., stimulus type). 


In [16]:
events_from_annotations, events_dict = mne.events_from_annotations(raw=eeg_data)
events_from_annotations = events_from_annotations[2:]

Used Annotations descriptions: ['New Segment/', 'Stimulus/S 54', 'Stimulus/S255']


## **1.4. Epoching the Data**

**Compulsory:** students write here the functional significance of this step (that is, why we do it). Lior's answer: In this step, we cut the continuous EEG recording into short time windows (epochs) centered around each TMS pulse. Epoching is necessary to segment the data around specific experimental events, allowing us to average the responses (TEP) and isolate or process the TMS pulse artifact.

In [24]:
pre_interpolation_epochs = mne.Epochs(raw=eeg_data,
                                      events=events_from_annotations,
                                      tmin=-1.1,
                                      tmax=0.5,
                                      baseline=None)
pre_interpolation_epochs_tep = pre_interpolation_epochs.average()
pre_interpolation_epochs_tep.plot();

Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated


In [21]:
post_interpolation_eeg = mne.io.read_raw(fname="data/post_2_5_interpolation_eeg.fif",
                                         preload=True)
post_interpolation_epochs = mne.Epochs(raw=post_interpolation_eeg,
                                       events=events_from_annotations,
                                       tmin=-1.1,
                                       tmax=0.5,
                                       baseline=None)
post_interpolation_tep = post_interpolation_epochs.average()
post_interpolation_tep.plot();

Opening raw data file data/post_2_5_interpolation_eeg.fif...
    Range : 0 ... 3307599 =      0.000 ...   661.520 secs
Ready.
Reading 0 ... 3307599  =      0.000 ...   661.520 secs...
Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated


## **1.5.Filtering**

**Compulsory:** students write here the functional significance of this step. Lior's answer: We apply a high-pass filter (at 0.1 Hz) to remove very low-frequency noise and slow drifts, such as those caused by sweating. This step stabilizes the baseline of the EEG signal and retains the relevant frequencies for cortical activity analysis.

In [23]:
post_filtering_eeg = mne.filter.filter_data(data=post_interpolation_eeg.get_data(),
                                            sfreq=post_interpolation_eeg.info["sfreq"],
                                            l_freq=0.1,
                                            h_freq=None,
                                            method="iir",
                                            iir_params=None,
                                            copy=True,
                                            phase="zero")
post_filtering_eeg = mne.io.RawArray(data=post_filtering_eeg,
                                     info=post_interpolation_eeg.info)
post_filtering_epochs = mne.Epochs(raw=post_filtering_eeg,
                                   events=events_from_annotations,
                                   tmin=-1.1,
                                   tmax=0.5,
                                   baseline=None)
post_filtering_tep = post_filtering_epochs.average()
post_filtering_tep.plot();

Setting up high-pass filter at 0.1 Hz

IIR filter parameters
---------------------
Butterworth highpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 8 (effective, after forward-backward)
- Cutoff at 0.10 Hz: -6.02 dB

Creating RawArray with float64 data, n_channels=70, n_times=3307600
    Range : 0 ... 3307599 =      0.000 ...   661.520 secs
Ready.
Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated
